# Energy data review

Inspect **one** built energy product at a time: its summary, interactive map, validation report and (for inferred products) the VIIRS nightlight input.

Dev-only: reads the pipeline's standard outputs; the packaged `energy` model stays visualisation-free. Build products first (see `../01-build-network`).

Run the first code cell to list the available products, then set `PRODUCT`.

In [ ]:
import pathlib
import sys

for _candidate in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_candidate / "_helpers.py").exists():
        sys.path.insert(0, str(_candidate))
        break
# _helpers is a dev-only module (reads pipeline outputs, not the energy package)
import _helpers as h  # noqa: E402

products = h.available_products()
print("Built products:")
display(h.list_products())
for i, name in enumerate(products):
    print(f"  [{i}] {name}")

# >>> pick which product to inspect (change the index) <<<
PRODUCT = products[0]
print("\nSelected PRODUCT =", PRODUCT)

## Summary

In [ ]:
nodes, edges = h.load_layers(PRODUCT)
print(f"{PRODUCT}: {len(nodes)} nodes, {len(edges)} edges, CRS EPSG:{edges.crs.to_epsg()}")
edges["source"].value_counts(dropna=False)

## Interactive map

Pan, zoom and **hover** over buses for attributes (Plotly on an OpenStreetMap basemap — no API key). Every edge is drawn, one colour per source (tab10): for the inferred products the dense OSM road mesh *is* the distribution network, drawn under the transmission / backbone / anchor layers. Only substation and generator buses are marked. Pass `roads=False` for a power-only view, or `clip=h.MAURITIUS_BBOX` to focus one island. `h.plot_network(PRODUCT)` gives a static matplotlib version.

In [ ]:
h.explore_network(PRODUCT)

## Validation report

In [ ]:
h.load_validation(PRODUCT)

## Nightlight raster (inferred input)

The inferred products retain OSM roads near VIIRS nightlight targets. Preview the composite if present:

In [ ]:
import rioxarray

tif = h.DATA_ROOT / "incoming/energy/nightlights/viirs-mauritius-rodrigues-2024.tif"
if tif.exists():
    da = rioxarray.open_rasterio(tif, masked=True).squeeze()
    da.plot.imshow(robust=True, figsize=(9, 7))
else:
    print("nightlight composite not found:", tif)